# Interactive Shape Blend Splines Demo (Paper-Faithful)

This notebook demonstrates the core design message of Li (2011):

- Use **4 control points**.
- Use SBS **partition-of-unity basis functions** built from smooth piecewise-polynomial step functions.
- Vary scalar weights \(w_0,\dots,w_3\) to produce different closed curves.

\[
\mathbf{C}(t)=\frac{\sum_{j=0}^{3} w_j\,W_j(t)\,\mathbf{P}_j}{\sum_{j=0}^{3} w_j\,W_j(t)},\qquad t\in[0,1)
\]

The basis construction is polynomial-only: **no \(\cos\), no \(\sin\), no rational basis kernels.**


In [ ]:
# Colab setup (safe to ignore outside Colab)
# %pip install -q shape-blend-splines ipywidgets matplotlib numpy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from shape_blend_splines.basis import (
    recursive_smooth_step,
    smooth_step_at,
    sbs_basis,
    uniform_bspline_weights,
)


## 1) SBS basis functions (the key innovation)

For an interval \([a,b]\), define smooth step functions centered at the endpoints and set

\[
B_{a,b}(t)=S_b(t)-S_a(t).
\]

Using these step differences, we build compact, smooth, partition-of-unity weights that are **piecewise polynomials**.


In [ ]:
def circular_midpoint(a, b, period=1.0):
    a = float(a)
    b = float(b)
    if b < a:
        b += period
    return (0.5 * (a + b)) % period


def periodic_blend_weights(t, centers, locality=1.0, order=2, period=1.0):
    # Periodic SBS partition-of-unity weights built from smooth step differences.
    t = np.asarray(t, dtype=float) % period
    centers = np.asarray(centers, dtype=float) % period
    k = len(centers)
    if k == 0:
        raise ValueError("centers must be non-empty")
    if locality < 0:
        raise ValueError("locality must be non-negative")
    if k == 1:
        return np.ones((1, len(t)), dtype=float)
    if locality == 0:
        return np.full((k, len(t)), 1.0 / k, dtype=float)

    raw = np.zeros((k, len(t)), dtype=float)
    for j in range(k):
        c_prev = centers[(j - 1) % k]
        c_curr = centers[j]
        c_next = centers[(j + 1) % k]

        left = circular_midpoint(c_prev, c_curr, period=period)
        right = circular_midpoint(c_curr, c_next, period=period)

        t_local = t.copy()
        if right <= left:
            right += period
            t_local = np.where(t_local < left, t_local + period, t_local)

        base_half_width = 0.5 * (right - left)
        half_width = max(base_half_width / float(locality), 1e-12)
        raw[j] = sbs_basis(t_local, left, right, half_width=half_width, order=order)

    total = raw.sum(axis=0, keepdims=True)
    total = np.where(total < 1e-12, 1.0, total)
    return raw / total


def rational_sbs_curve(t, control_points, point_weights, locality=2.0, order=2):
    centers = np.arange(len(control_points), dtype=float) / len(control_points)
    W = periodic_blend_weights(t, centers, locality=locality, order=order)
    rw = point_weights[:, None] * W
    denom = rw.sum(axis=0)
    denom = np.where(denom < 1e-12, 1.0, denom)
    num = np.einsum('jm,jd->md', rw, control_points)
    return num / denom[:, None], W


In [ ]:
# Step-construction sketch: S_a, S_b, and B_{a,b}=S_b-S_a

t_plot = np.linspace(-0.1, 1.1, 600)
a, b = 0.25, 0.75
h = 0.2

S_a = smooth_step_at(t_plot, a, h, order=2, rising=False)
S_b = smooth_step_at(t_plot, b, h, order=2, rising=False)
B_ab = np.clip(S_b - S_a, 0.0, None)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t_plot, S_a, lw=2, label=r'$S_a(t)$')
ax.plot(t_plot, S_b, lw=2, label=r'$S_b(t)$')
ax.plot(t_plot, B_ab, lw=3, color='black', label=r'$B_{a,b}(t)=S_b-S_a$')
ax.axvline(a, color='gray', ls='--', lw=1)
ax.axvline(b, color='gray', ls='--', lw=1)
ax.set_xlabel('t')
ax.set_ylabel('value')
ax.set_ylim(-0.05, 1.05)
ax.set_title('SBS basis from two smooth polynomial steps')
ax.legend(loc='upper center', ncol=3)
plt.show()


In [ ]:
# Four periodic SBS basis functions at different smoothness orders

t_basis = np.linspace(0.0, 1.0, 1000, endpoint=False)
centers_4 = np.array([0.0, 0.25, 0.5, 0.75])
orders = [1, 2, 3]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, ord_ in zip(axes, orders):
    W = periodic_blend_weights(t_basis, centers_4, locality=2.5, order=ord_)
    for j in range(4):
        ax.plot(t_basis, W[j], lw=2, label=fr'$B_{j}(t)$')
    ax.plot(t_basis, W.sum(axis=0), 'k--', lw=1.5, label=r'$\sum_j W_j(t)$')
    ax.set_title(f'order = {ord_}')
    ax.set_xlabel('t')
    ax.set_ylim(-0.02, 1.08)
axes[0].set_ylabel('basis value')
axes[0].legend(loc='upper right', fontsize=8)
fig.suptitle('Periodic SBS basis functions (partition of unity)', y=1.03)
plt.tight_layout()
plt.show()


## 2) Four control points \(\rightarrow\) different closed curves

We keep the same control points:

\[
\mathbf{P}_0=(1,0),\;\mathbf{P}_1=(0,1),\;\mathbf{P}_2=(-1,0),\;\mathbf{P}_3=(0,-1)
\]

and vary \(w_j\) (and \(\alpha\), the locality parameter) to get different curves.


In [ ]:
control_points = np.array([
    [ 1.0,  0.0],
    [ 0.0,  1.0],
    [-1.0,  0.0],
    [ 0.0, -1.0],
], dtype=float)

panel_configs = [
    (np.array([1.0, 1.0, 1.0, 1.0]), 1.2, 'Rounded / circle-like'),
    (np.array([1.8, 1.0, 1.8, 1.0]), 2.8, 'Intermediate (elliptic tendency)'),
    (np.array([1.0, 1.0, 1.0, 1.0]), 8.0, 'Near-square (high locality)'),
]

t = np.linspace(0.0, 1.0, 1200, endpoint=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (w, alpha, title) in zip(axes, panel_configs):
    pts, _ = rational_sbs_curve(t, control_points, w, locality=alpha, order=2)
    ax.plot(pts[:, 0], pts[:, 1], color='tab:blue', lw=2.5)
    ax.plot(*control_points.T, 'ko', ms=5)
    ax.plot([control_points[-1, 0], control_points[0, 0]],
            [control_points[-1, 1], control_points[0, 1]],
            'k--', lw=1, alpha=0.45)
    ax.plot(control_points[:, 0], control_points[:, 1], 'k--', lw=1, alpha=0.45)

    for j, p in enumerate(control_points):
        ax.text(p[0] + 0.05, p[1] + 0.05, f'P{j}', fontsize=9)
        ax.text(p[0] + 0.05, p[1] - 0.12, f'w_{j}={w[j]:.1f}', fontsize=9, color='tab:red')

    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    ax.set_title(f'{title}\n$\\alpha={alpha:.1f}$')
    ax.grid(alpha=0.2)

fig.suptitle('Same 4 control points; different weights/locality produce different closed curves', y=1.02)
plt.tight_layout()
plt.show()


## 3) Interactive weight sliders

Move \(w_0,\dots,w_3\) and \(\alpha\) to see how the same four control points generate a family of closed curves.


In [ ]:
def interactive_plot(w0=1.0, w1=1.0, w2=1.0, w3=1.0, alpha=3.0):
    w = np.array([w0, w1, w2, w3], dtype=float)
    t = np.linspace(0.0, 1.0, 1200, endpoint=False)
    pts, _ = rational_sbs_curve(t, control_points, w, locality=alpha, order=2)

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.plot(pts[:, 0], pts[:, 1], color='tab:blue', lw=2.5)
    ax.plot(control_points[:, 0], control_points[:, 1], 'ko', ms=5)
    ax.plot([control_points[-1, 0], control_points[0, 0]],
            [control_points[-1, 1], control_points[0, 1]],
            'k--', lw=1, alpha=0.45)
    ax.plot(control_points[:, 0], control_points[:, 1], 'k--', lw=1, alpha=0.45)

    for j, p in enumerate(control_points):
        ax.text(p[0] + 0.05, p[1] + 0.05, f'P{j}', fontsize=10)
        ax.text(p[0] + 0.05, p[1] - 0.14, f'w_{j}={w[j]:.2f}', fontsize=10, color='tab:red')

    ax.set_title(rf'Rational SBS curve   ($\\alpha={alpha:.2f}$)')
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(-1.35, 1.35)
    ax.set_ylim(-1.35, 1.35)
    ax.grid(alpha=0.25)
    plt.show()

widgets.interact(
    interactive_plot,
    w0=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='w0'),
    w1=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='w1'),
    w2=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='w2'),
    w3=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='w3'),
    alpha=widgets.FloatSlider(value=3.0, min=0.5, max=12.0, step=0.1, description='α'),
)


## 4) Basis visualization vs locality \(\alpha\)

Larger \(\alpha\) narrows transition bands and increases locality.

**These basis functions are piecewise polynomials — no trigonometric or rational basis kernels are used.**


In [ ]:
t_basis = np.linspace(0.0, 1.0, 1000, endpoint=False)
centers_4 = np.array([0.0, 0.25, 0.5, 0.75])

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, alpha in zip(axes, [1.0, 3.0, 8.0]):
    W = periodic_blend_weights(t_basis, centers_4, locality=alpha, order=2)
    for j in range(4):
        ax.plot(t_basis, W[j], lw=2, label=fr'$W_{j}(t)$')
    ax.set_title(rf'$\\alpha={alpha}$')
    ax.set_xlabel('t')
    ax.set_ylim(-0.02, 1.08)
axes[0].set_ylabel('weight value')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()


## 5) Comparison with cubic B-spline basis

SBS and cubic B-spline basis functions both provide local, smooth, partition-of-unity behavior.

SBS here is constructed from smooth polynomial step functions; cubic B-spline is shown as a familiar reference.


In [ ]:
t_cmp = np.linspace(0.0, 1.0, 600)
W_sbs = periodic_blend_weights(t_cmp[:-1], centers_4, locality=3.0, order=2)
W_bs = uniform_bspline_weights(t_cmp, n=4, degree=3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for j in range(4):
    axes[0].plot(t_cmp[:-1], W_sbs[j], lw=2, label=fr'$W_{j}(t)$')
axes[0].set_title('SBS basis (smooth polynomial steps)')
axes[0].set_xlabel('t')
axes[0].set_ylabel('basis value')
axes[0].set_ylim(-0.02, 1.08)
axes[0].legend(fontsize=8)

for j in range(4):
    axes[1].plot(t_cmp, W_bs[j], lw=2, label=fr'$N_{j,3}(t)$')
axes[1].set_title('Cubic B-spline basis (reference)')
axes[1].set_xlabel('t')
axes[1].set_ylim(-0.02, 1.08)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
